# PN25 — corrected pair-ridge compression

## TL;DR

The corrected odds-to-ARA coordinate is exact, and three reversible pair classes preserve essentially all tested
information in the six mod-14 lanes. On 6,000 prospective anchors across three scales, however, ridge-closeness did
not predict fewer handovers, immediate prime closure, three-state closure, or upward path movement. Status:
**geometric-only support / dynamic null**.


## Context & Methods

For mod-14 anti-pair `(a,14-a)`, directional odds `q=a/(14-a)` convert to TE-ARA share
`x_A=2q/(1+q)=a/7`, with `x_B=2-x_A`. Pair-closeness is
`c=min(r,14-r)/7`; orientation is the sign of `r-7`.

The PN24 opened sample supplies frozen model rates. Three new 2,000-anchor ranges are scored without refitting.
The protected 87-bit anchor is absent.


In [1]:
import csv
import json
from pathlib import Path

HERE = Path.cwd()
results = json.loads((HERE / 'PN25_PAIR_RIDGE_COMPRESSION_RESULTS.json').read_text(encoding='utf-8'))
validation = json.loads((HERE / 'PN25_PAIR_RIDGE_COMPRESSION_VALIDATION.json').read_text(encoding='utf-8'))
with (HERE / 'PN25_PAIR_RIDGE_COMPRESSION_TARGETS.csv').open(encoding='utf-8', newline='') as handle:
    targets = list(csv.DictReader(handle))
print(results['status'])
print('validation:', validation['status'], validation['checks_passed'], '/', validation['checks_total'])
assert results['status'] == 'GEOMETRIC-ONLY SUPPORT / DYNAMIC NULL'
assert validation['status'] == 'PASS'
assert len(targets) == 6000


GEOMETRIC-ONLY SUPPORT / DYNAMIC NULL
validation: PASS 14 / 14


## Data

In [2]:
print(results['data'])
for scale in ('low', 'middle', 'high'):
    assert sum(row['scale'] == scale for row in targets) == 2000
assert results['data']['protected_87_bit_anchor_used'] is False


{'development_source': 'PN24_NEAREST_HANDOVER_CASCADE_ANCHORS.csv', 'development_n': 2000, 'target_ranges': [{'scale': 'low', 'low': 61000000, 'high_exclusive': 61500000, 'seed': 25001, 'n': 2000}, {'scale': 'middle', 'low': 61000000000, 'high_exclusive': 61000500000, 'seed': 25002, 'n': 2000}, {'scale': 'high', 'low': 610000000000, 'high_exclusive': 610000500000, 'seed': 25003, 'n': 2000}], 'target_n': 6000, 'target_distinct_next_prime_labels': 5536, 'protected_87_bit_anchor_used': False}


## Results — exact coordinate

In [3]:
for row in results['exact_coordinate_checks']:
    print(row['pair'], row['odds'], '->', row['converted_A'], '+', row['converted_B'])
    assert row['conversion_exact'] is True
    assert row['te_ara_sum_exact'] is True
assert results['exact_coordinate_pass'] is True


[1, 13] 1/13 -> 1/7 + 13/7
[3, 11] 3/11 -> 3/7 + 11/7
[5, 9] 5/9 -> 5/7 + 9/7
[7, 7] 1 -> 1 + 1


## Results — prospective handover predictions

In [4]:
print('scale | mean H by 1/7,3/7,5/7 | Y0 rates | Y3 rates')
for scale, row in results['ordering_checks'].items():
    print(scale, row['mean_handovers'], row['Y0_rates'], row['Y3_rates'])
print('scale correlations:', results['scale_correlations_c_vs_H'])
print('permutation:', results['permutation_test'])
print('path progression:', results['path_progression'])
print('prediction verdicts:', results['predictions'])
assert results['predictions']['dynamic_predictions_passed'] == 0


scale | mean H by 1/7,3/7,5/7 | Y0 rates | Y3 rates
low [1.9256637168141593, 2.0, 1.9312796208530805] [0.1256637168141593, 0.1218274111675127, 0.1338862559241706] [0.7026548672566372, 0.6751269035532995, 0.6966824644549763]
middle [2.2327586206896552, 2.196521739130435, 2.2650887573964495] [0.07758620689655173, 0.10782608695652174, 0.07692307692307693] [0.6362068965517241, 0.6295652173913043, 0.6189349112426036]
high [2.3150442477876108, 2.3783783783783785, 2.3226571767497033] [0.07964601769911504, 0.09797297297297297, 0.09727164887307237] [0.5823008849557522, 0.5692567567567568, 0.5907473309608541]
pooled [2.1584795321637427, 2.1916951080773606, 2.1729857819905214] [0.09415204678362574, 0.10921501706484642, 0.10268562401263823] [0.6403508771929824, 0.6245733788395904, 0.6354660347551343]
scale correlations: {'low': -0.000919871093350331, 'middle': 0.011323499544581032, 'high': 0.0001263401464240898}
permutation: {'observed_pearson_c_vs_H': 0.0033352855563471254, 'alternative': 'negati

## Results — pair compression versus six lanes

In [5]:
for outcome, scores in results['compression_scores'].items():
    print(outcome, scores)
    assert scores['pair_within_2_percent_of_lane'] is True
    assert scores['pair_beats_global'] is False
    assert scores['lane_beats_global'] is False
assert results['compression_fidelity_pass'] is True


Y0 {'global_brier': 0.09192, 'orientation_brier': 0.09198958296128329, 'pair_brier': 0.09204273466071529, 'lane_brier': 0.09208748126657003, 'pair_relative_loss_vs_lane': -0.0004859141029735576, 'pair_within_2_percent_of_lane': True, 'pair_beats_global': False, 'lane_beats_global': False}
Y3 {'global_brier': 0.23214124999999997, 'orientation_brier': 0.23209361441572185, 'pair_brier': 0.23222937287414352, 'lane_brier': 0.23223524856284425, 'pair_relative_loss_vs_lane': -2.530058954049717e-05, 'pair_within_2_percent_of_lane': True, 'pair_beats_global': False, 'lane_beats_global': False}


## Takeaways

1. `(1,13)`, `(3,11)` and `(5,9)` are three complete pair identities at different lateral compositions; they are
   not three allocations to add.
2. Odds convert exactly to the bounded total-2 ARA coordinate: `1/13 -> 1/7`, `3/11 -> 3/7`, `5/9 -> 5/7`,
   with the excluded `7/7 -> 1` ridge.
3. Pair-closeness plus orientation reconstructs all six mod-14 lanes exactly. Discarding orientation preserved the
   tested outcome scores to much better than the frozen 2% tolerance.
4. Pair-closeness did not order future prime handovers. The pooled correlation was `+0.003335`, with one-sided
   permutation `p=0.6110` for the predicted negative relation.
5. The pair coordinate is therefore lateral wheel geometry. Higher prime gates form a separate vertical state that
   remains necessary for next-prime completion.
